# 🧮 Clase 01 · Matrices con NumPy — la imagen es una matriz

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pdavicho/algebra-lineal-uees/blob/main/notebooks/clase-01.ipynb)

**Álgebra Lineal · UMAT205 · UEES** — práctica de la Clase 01 (Semana 1).

En este notebook vas a:
1. Crear matrices con **NumPy** y consultar su dimensión y sus elementos $a_{ij}$.
2. Hacer las operaciones de la clase: suma, resta, escalar y **producto de matrices**.
3. Resolver el reto oficial de la semana, **"La imagen es una matriz"**: brillo,
   blending, detección de movimiento e **inpainting con eliminación de Gauss-Jordan**.

> ▶ Ejecuta cada celda con `Shift + Enter` (o el botón de play). No necesitas instalar nada.


## 1. Tu primera matriz

En Python, la librería **NumPy** es el estándar para trabajar con matrices
(la usan desde la NASA hasta los creadores de ChatGPT). Un arreglo de NumPy
se crea con una *lista de listas*: cada lista interior es una **fila**.


In [ ]:
import numpy as np

A = np.array([[2, -1, 0],
              [4,  5, 3]])

print(A)
print("Dimensión (filas, columnas):", A.shape)   # m × n
print("Número total de elementos:", A.size)


### El elemento $a_{ij}$

En matemáticas contamos desde 1 ($a_{11}$ es la esquina superior izquierda),
pero **Python cuenta desde 0**. Así que el elemento $a_{23}$ (fila 2, columna 3)
se escribe `A[1, 2]`.

⚠️ Este desfase es la causa #1 de errores al programar álgebra lineal. ¡Grábatelo!


In [ ]:
print("a_11 =", A[0, 0])   # fila 1, columna 1
print("a_23 =", A[1, 2])   # fila 2, columna 3

# Una fila o columna completas:
print("Fila 2 completa:   ", A[1, :])
print("Columna 3 completa:", A[:, 2])


### 🎯 Reto 1
Crea una matriz `B` de dimensión $3 \times 2$ con los números que quieras
e imprime su elemento $b_{31}$ (¡recuerda el desfase de índices!).


In [ ]:
# Tu código aquí 👇


## 2. Matrices especiales

NumPy trae "fábricas" de las matrices especiales que vimos en clase:


In [ ]:
I = np.eye(3)          # identidad 3×3
O = np.zeros((2, 3))   # nula 2×3
D = np.diag([7, 2, 5]) # diagonal con 7, 2, 5

print("Identidad I3:\n", I)
print("\nNula 2×3:\n", O)
print("\nDiagonal:\n", D)


## 3. Operaciones con matrices

### Suma, resta y escalar — elemento a elemento


In [ ]:
A = np.array([[3, -1],
              [2,  5]])
B = np.array([[1,  4],
              [-2, 0]])

print("A + B =\n", A + B)
print("\nA - B =\n", A - B)
print("\n3·A =\n", 3 * A)


### Producto de matrices: el operador `@`

En NumPy el producto **fila × columna** de la clase se escribe con `@`.

⚠️ Cuidado: `A * B` (asterisco) multiplica *elemento a elemento* — **NO** es el
producto de matrices. Ese error ha arruinado más de una tarea…


In [ ]:
A = np.array([[2, 1],
              [0, 3]])
B = np.array([[1, 4],
              [5, 2]])

print("Producto de matrices A @ B =\n", A @ B)
print("\nElemento a elemento A * B (¡NO es el producto de matrices!) =\n", A * B)


### El producto NO es conmutativo

Comprobemos con código lo que vimos en la demo de la clase:


In [ ]:
print("A @ B =\n", A @ B)
print("\nB @ A =\n", B @ A)
print("\n¿A@B es igual a B@A?:", np.array_equal(A @ B, B @ A))

# Y la identidad sí es el neutro:
I = np.eye(2)
print("\nA @ I =\n", A @ I)


### ¿Y si las dimensiones no encajan?

Si las columnas de $A$ no coinciden con las filas de $B$, NumPy lanza un error.
Ejecuta la celda y **lee el mensaje**: entender errores es parte de programar.


In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])      # 2×3
B = np.array([[1, 2, 3],
              [4, 5, 6]])      # 2×3  ← ¡filas de B (2) ≠ columnas de A (3)!

try:
    A @ B
except ValueError as e:
    print("💥 Error de NumPy:", e)
    print("\nTraducción: (2×3)@(2×3) no existe. Necesitamos que B sea 3×p.")

print("\nSolución: usar la transpuesta B.T, que es", B.T.shape, ":")
print(A @ B.T)   # (2×3)@(3×2) = 2×2 ✔


## 4. 📸 Una imagen ES una matriz

Momento estrella: vamos a cargar una foto real y verla como lo que es — una matriz.
Usaremos la clásica foto del *camarógrafo* que viene incluida en `scikit-image`.


In [ ]:
from skimage import data
import matplotlib.pyplot as plt

foto = data.camera()   # imagen en escala de grises

print("Tipo:", type(foto))
print("¡La foto es una matriz de dimensión", foto.shape, "!")
print("Cada elemento va de 0 (negro) a 255 (blanco).")
print("\nEsquina superior izquierda (submatriz 5×5):\n", foto[:5, :5])

plt.imshow(foto, cmap="gray", vmin=0, vmax=255)
plt.title(f"Esto es una matriz {foto.shape[0]}×{foto.shape[1]}")
plt.axis("off")
plt.show()


### Editar la foto = operar con la matriz

Cada operación de la clase tiene un efecto visual:

| Álgebra | Efecto en la foto |
|---|---|
| $1.5 \cdot A$ (escalar) | más brillo |
| $255 - A$ (resta) | negativo |
| $A^T$ (transpuesta) | girar/reflejar |
| submatriz $A[i_1{:}i_2,\ j_1{:}j_2]$ | recortar |


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

brillo = np.clip(foto * 1.5, 0, 255)       # escalar: k·A (clip evita pasar de 255)
negativo = 255 - foto                       # resta de matrices
transpuesta = foto.T                        # transpuesta
recorte = foto[50:250, 150:350]             # submatriz (el rostro)

for ax, img, titulo in zip(
    axes,
    [brillo, negativo, transpuesta, recorte],
    ["1.5·A → brillo", "255−A → negativo", "Aᵀ → transpuesta", "Submatriz → recorte"],
):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(titulo)
    ax.axis("off")

plt.show()


### 🎯 Reto 2
1. Oscurece la foto multiplicándola por un escalar menor que 1.
2. Recorta la cámara que sostiene el camarógrafo (juega con los índices de la submatriz).
3. **Bonus:** ¿qué pasa si haces `(foto + negativo)`? ¿Por qué da una imagen blanco uniforme?
   Explícalo con álgebra de matrices.


In [ ]:
# Tu código aquí 👇




## 5. 🎨 Blending: mezclar dos imágenes con una suma ponderada

"Blending" es el efecto de disolución entre dos fotos (fundidos de video, marcas
de agua semitransparentes, filtros). Álgebra pura: si $A$ y $B$ son dos imágenes
de **la misma dimensión**, la mezcla con un factor $\alpha \in [0,1]$ es

$$C = \alpha \cdot A + (1-\alpha)\cdot B$$

— una multiplicación por escalar de cada imagen, más una suma de matrices. Con
$\alpha=1$ ves solo $A$; con $\alpha=0$ ves solo $B$; en el medio, la mezcla.


In [ ]:
from skimage.transform import resize

foto_A = data.camera().astype(float)                 # imagen 1: el camarógrafo
foto_B = resize(data.moon(), foto_A.shape,            # imagen 2: la luna
                 preserve_range=True, anti_aliasing=True)

print("Dimensión de A:", foto_A.shape, "· Dimensión de B:", foto_B.shape)
print("Deben coincidir para poder sumarlas — por eso se hizo resize() sobre B.")

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
alphas = [1.0, 0.75, 0.5, 0.25, 0.0]

for ax, alpha in zip(axes, alphas):
    mezcla = alpha * foto_A + (1 - alpha) * foto_B    # C = α·A + (1-α)·B
    ax.imshow(np.clip(mezcla, 0, 255), cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"α = {alpha}")
    ax.axis("off")

plt.suptitle("Blending: C = α·A + (1−α)·B — escalar + suma de matrices")
plt.show()


## 6. 🏃 Detección de movimiento: la resta de dos fotogramas

Una cámara de seguridad "sabe" que algo se movió con una sola operación:
**restar** el fotograma actual del anterior. Donde nada cambió, la resta da
(casi) cero — negro. Donde algo se movió, la diferencia es grande — se ve
brillante.

$$\text{movimiento} = |\,F_{t} - F_{t-1}\,|$$

No tenemos un video a mano, así que **simulamos** un segundo fotograma
desplazando la foto unos píxeles (`np.roll`) — como si algo dentro de la
escena se hubiera corrido. En un video real, $F_{t-1}$ y $F_t$ serían dos
fotogramas consecutivos de verdad, pero la operación de álgebra es exactamente
la misma.


In [ ]:
frame_anterior = data.camera().astype(float)
# "Frame actual": la misma escena, con el camarógrafo unos píxeles más a la derecha/abajo.
frame_actual = np.roll(frame_anterior, shift=(8, 12), axis=(0, 1))

diferencia = np.abs(frame_actual - frame_anterior)          # resta de matrices + valor absoluto
umbral = 25
movimiento = diferencia > umbral                             # máscara: ¿dónde hubo cambio real?

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, titulo, cmap in zip(
    axes,
    [frame_anterior, frame_actual, diferencia, movimiento],
    ["Frame t-1", "Frame t (simulado)", "|Ft − Ft-1|", f"Movimiento (> {umbral})"],
    ["gray", "gray", "gray", "gray"],
):
    ax.imshow(img, cmap=cmap, vmin=0, vmax=(255 if img.dtype != bool else 1))
    ax.set_title(titulo)
    ax.axis("off")

plt.show()
print("Porcentaje de píxeles con movimiento detectado:",
      f"{100 * movimiento.mean():.1f}%")


## 7. 🧩 Inpainting: reconstruir píxeles perdidos con Gauss-Jordan

Esta es la pieza que conecta directamente con el **tema 1.4** de hoy: eliminación
de Gauss-Jordan. La idea (revisa el ejemplo mínimo de 2 incógnitas en tu material
de repaso si lo tienes a mano) es de **suavidad**: un píxel desconocido debería
parecerse al promedio de sus vecinos.

- Si solo falta **un** píxel, su valor sale directo del promedio de sus vecinos
  — ni siquiera hace falta resolver un sistema.
- Si falta un **hueco** de varios píxeles seguidos, cada incógnita depende de
  OTRAS incógnitas del mismo hueco (sus vecinos también pueden estar perdidos).
  Ahí es donde se necesita un **sistema de ecuaciones lineales** — y Gauss-Jordan.

Para un píxel desconocido $x_{i,j}$ con 4 vecinos (arriba, abajo, izquierda,
derecha), la condición de suavidad "cada píxel es el promedio de sus vecinos" se
escribe como ecuación lineal:

$$4\,x_{i,j} - x_{i-1,j} - x_{i+1,j} - x_{i,j-1} - x_{i,j+1} = 0$$

Una ecuación de estas **por cada píxel perdido** arma el sistema $Ax=b$ completo
(los vecinos ya conocidos se mueven al lado derecho, $b$).

### 7.1 Primero, a mano: nuestra propia función `gauss_jordan`

Antes de aplicarlo a una imagen real, programamos el método tal cual lo vimos en
la demo de la clase — con pivoteo parcial — y lo probamos con el mismo sistema
pequeño de 2 incógnitas del ejemplo de repaso.


In [ ]:
def gauss_jordan(A, b):
    """Resuelve A x = b reduciendo la matriz ampliada [A|b] a su forma
    escalonada reducida (RREF), con pivoteo parcial. Devuelve el vector x."""
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)
    M = np.hstack([A, b.reshape(-1, 1)])          # matriz ampliada [A|b]

    for col in range(n):
        piv = np.argmax(np.abs(M[col:, col])) + col   # fila con mayor |valor| en esta columna
        if piv != col:
            M[[col, piv]] = M[[piv, col]]              # F_col <-> F_piv
        M[col] = M[col] / M[col, col]                  # normaliza el pivote a 1
        for r in range(n):
            if r != col:
                M[r] = M[r] - M[r, col] * M[col]       # hace 0 arriba y abajo del pivote

    return M[:, -1]


# Verificación con el sistema de 2 incógnitas del material de repaso:
#   2x1 -  x2 = 120
#  -x1  + 2x2 = 140
A_prueba = [[2, -1], [-1, 2]]
b_prueba = [120, 140]
x_prueba = gauss_jordan(A_prueba, b_prueba)
print("Nuestra gauss_jordan:", x_prueba)
print("np.linalg.solve     :", np.linalg.solve(A_prueba, b_prueba))
print("(deben coincidir, y parecerse a x1≈126.7, x2≈133.3)")


### 7.2 Ahora a lo grande: reconstruir un hueco real de la foto

Vamos a "dañar" un hueco cuadrado de la foto (borrarlo) y reconstruirlo
resolviendo un sistema — una incógnita por cada píxel perdido. Con un hueco de
$h \times w$ píxeles el sistema tiene $h\cdot w$ incógnitas: para un hueco de
10×10 son **100 incógnitas acopladas entre sí**, exactamente el caso en que
hace falta Gauss-Jordan (y no un simple promedio).


In [ ]:
import time
import matplotlib.patches as patches

foto_original = data.camera().astype(float)

# --- 1. "Dañamos" un hueco cuadrado h×w en la posición (r0, c0) ---
r0, c0, h, w = 195, 270, 10, 10
foto_danada = foto_original.copy()
foto_danada[r0:r0 + h, c0:c0 + w] = np.nan   # NaN = "no sabemos este píxel"

# --- 2. Armamos el sistema A x = b: una incógnita por píxel del hueco ---
n = h * w
idx = lambda i, j: i * w + j              # índice de la incógnita (i,j) dentro del hueco
A = np.zeros((n, n))
b = np.zeros(n)

for i in range(h):
    for j in range(w):
        k = idx(i, j)
        A[k, k] = 4                        # 4·x_ij ...
        for (vi, vj) in [(i - 1, j), (i + 1, j), (i, j - 1), (i, j + 1)]:
            if 0 <= vi < h and 0 <= vj < w:
                A[k, idx(vi, vj)] = -1      # ... - vecino (si también es incógnita) ...
            else:
                fi, fj = r0 + vi, c0 + vj   # ... o el valor CONOCIDO se va a b
                b[k] += foto_original[fi, fj]

print(f"Sistema de {n} incógnitas (hueco de {h}×{w} píxeles).")

# --- 3. Resolvemos con NUESTRA gauss_jordan, y comparamos con np.linalg.solve ---
t0 = time.time()
x_mano = gauss_jordan(A, b)
t_mano = time.time() - t0

t0 = time.time()
x_numpy = np.linalg.solve(A, b)
t_numpy = time.time() - t0

print(f"gauss_jordan (nuestra):  {t_mano*1000:.2f} ms")
print(f"np.linalg.solve:         {t_numpy*1000:.2f} ms")
print("¿Mismo resultado?", np.allclose(x_mano, x_numpy))

# --- 4. Reconstruimos la imagen con la solución ---
foto_reconstruida = foto_danada.copy()
foto_reconstruida[r0:r0 + h, c0:c0 + w] = x_mano.reshape(h, w)

# --- 5. Visualizamos ---
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, img, titulo in zip(
    axes,
    [foto_original, foto_danada, foto_reconstruida],
    ["Original", "Con hueco (NaN)", "Reconstruida con Gauss-Jordan"],
):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.add_patch(patches.Rectangle((c0, r0), w, h, edgecolor="red", facecolor="none", linewidth=1.5))
    ax.set_title(titulo)
    ax.axis("off")
plt.show()

error = np.abs(foto_reconstruida[r0:r0+h, c0:c0+w] - foto_original[r0:r0+h, c0:c0+w])
print(f"Error promedio vs. los píxeles reales (que sí conocíamos, para comprobar): {error.mean():.2f}")


### 🎯 Reto 3
1. Prueba con un hueco chico (`h=4, w=4`) y con uno grande (`h=20, w=20`) y
   compara los tiempos. Con pocas incógnitas, `gauss_jordan` puede verse
   "igual de rápido o más" que `np.linalg.solve` — el tiempo fijo de arranque
   de la librería domina. Con muchas incógnitas, la diferencia se dispara a
   favor de `np.linalg.solve`, porque nuestra implementación crece
   aproximadamente $O(n^3)$ mientras que LAPACK (la librería que usa NumPy por
   debajo) está optimizada a fondo. Anota con qué tamaño de hueco empieza a
   notarse la diferencia en tu máquina.
2. Mueve el hueco muy cerca del borde de la imagen (por ejemplo `r0 = 1`).
   **Ojo:** el código de la sección 7.2 solo revisa si un vecino cae fuera del
   *hueco*, no si cae fuera de *toda la imagen* — con índices negativos, NumPy
   no da error, "envuelve" y toma un píxel del otro extremo de la imagen (fila
   -1 = última fila). Encuentra ese error y corrígelo (pista: además de
   `0 <= vi < h and 0 <= vj < w`, valida que `0 <= r0+vi < foto_original.shape[0]`
   y lo mismo para las columnas).
3. **Bonus:** en vez de un hueco cuadrado, "daña" una franja delgada (por
   ejemplo `h=2, w=20`) y compara visualmente qué tan bien se reconstruye una
   forma alargada frente a una cuadrada.


In [ ]:
# Tu código aquí 👇




## 8. 🤖 Semilla para las próximas clases

Te dejo una pista de hacia dónde vamos. Un modelo de lenguaje guarda cada palabra
como un **vector** (una matriz fila). Palabras parecidas → vectores parecidos:


In [ ]:
# Mini-embeddings de juguete (2 dimensiones: [tamaño, "felinidad"])
palabras = {
    "gato":   np.array([0.2, 0.9]),
    "tigre":  np.array([0.8, 0.9]),
    "perro":  np.array([0.3, 0.1]),
}

for w, v in palabras.items():
    print(f"{w:6s} → {v}")

# ¿Quién se parece más a 'gato'? Medimos con producto punto (¡álgebra lineal!)
print("\ngato·tigre =", palabras["gato"] @ palabras["tigre"])
print("gato·perro =", palabras["gato"] @ palabras["perro"])
print("\n→ El modelo 'sabe' que gato se parece más a tigre que a perro.")
print("En la Clase 05 haremos esto con embeddings REALES de miles de dimensiones.")


## 9. 🔗 Reto de enlace — antes de la próxima clase

La próxima semana trabajamos con la **inversa** y el **determinante** de una matriz,
para calcular a mano los pesos de una regresión lineal ($w = (X^TX)^{-1}X^Ty$).

Con la matriz `A` de abajo (cuadrada, 2×2):

1. Intenta encontrar por ensayo y error una matriz `A_inv` tal que `A @ A_inv` dé
   (aproximadamente) la matriz identidad. Todavía NO uses ninguna fórmula ni
   `np.linalg.inv` — es a puro tanteo, jugando con los números.
2. Anota cuánto te costó encontrarla (o si te rendiste) y trae tu resultado —
   o tu conjetura de por qué es difícil — a la próxima clase. Ahí vamos a ver
   el método exacto para calcularla siempre, sin adivinar.


In [ ]:
A = np.array([[2, 1],
              [1, 1]])

# Tu código aquí 👇 — propón A_inv y comprueba con A @ A_inv


---
## ✅ Checklist de salida

- [ ] Sé crear matrices con `np.array` y leer su `.shape`.
- [ ] Entiendo el desfase de índices: $a_{ij}$ es `A[i-1, j-1]`.
- [ ] Distingo `A @ B` (producto de matrices) de `A * B` (elemento a elemento).
- [ ] Comprobé que el producto no es conmutativo.
- [ ] Edité una imagen real usando operaciones de matrices (brillo, negativo, transpuesta, recorte).
- [ ] Mezclé dos imágenes con blending: $C = \alpha A + (1-\alpha) B$.
- [ ] Detecté "movimiento" restando dos fotogramas y aplicando un umbral.
- [ ] Programé mi propia función `gauss_jordan` y comprobé que coincide con `np.linalg.solve`.
- [ ] Reconstruí un hueco de píxeles perdidos armando y resolviendo el sistema $Ax=b$ del inpainting.

**Tarea:** completa la actividad de la Unidad 1 en Blackboard (plazo: 7 días).

[⬅ Volver al sitio del curso](https://pdavicho.github.io/algebra-lineal-uees/clases/clase-01/index.html)
